# Facial Emotion Detection using FER and OpenCV

This notebook detects the **emotion on a human face** in an image using the `FER` library
(which uses a CNN model, optionally boosted with MTCNN face detection) and **OpenCV** for
image handling and drawing.

**Pipeline:**
1. Load an image
2. Detect the face and predict emotion probabilities
3. Draw a bounding box around the face
4. Label the box with the dominant emotion
5. Save and display the final image


## Step 1: Import Libraries

- **cv2 (OpenCV)** → used to read images, draw rectangles/text, and save the output image.
- **fer** → a pre-trained deep learning library specifically built for *Facial Expression Recognition*.
- **matplotlib.pyplot** → used to display the final image inside the notebook.

`!pip install fer` installs the FER library directly from the notebook (only needed once).


In [ ]:
# Import OpenCV library
import cv2

# Install the FER (Facial Expression Recognition) library
!pip install fer

# Import FER class from the fer library
from fer import FER

# matplotlib.pyplot is a collection of command-style functions that make matplotlib work like MATLAB
import matplotlib.pyplot as plt

## Step 2: Load the Input Image

`cv2.imread()` reads the image from disk into a NumPy array (in **BGR** format, not RGB —
this matters later when displaying with matplotlib).

> ⚠️ Update `imgloc` to point to your own image file path.


In [ ]:
# Path to input image
imgloc = '/content/WhatsApp Image 2024-04-27 at 2.02.32 PM.jpeg'

# Read the image from location and store it in variable 'input_image'
input_image = cv2.imread(imgloc)

## Step 3: Initialize the Emotion Detector

`FER(mtcnn=True)` creates the detector object.
- `mtcnn=True` tells FER to use **MTCNN (Multi-task Cascaded Convolutional Network)** for
  face detection, which is more accurate than the default OpenCV Haar Cascade — especially
  for angled or partially visible faces.


In [ ]:
# To use the more advanced MTCNN (Multi-Cascade Convolutional Network) face detector,
# pass mtcnn=True to FER
emo_detector = FER(mtcnn=True)

## Step 4: Detect Emotions in the Image

`detect_emotions()` does two things internally:
1. Finds face(s) in the image (bounding box coordinates).
2. Runs a CNN classifier on each detected face to output a **probability score for each
   of the 7 emotions**: angry, disgust, fear, happy, sad, surprise, neutral.

The output is a **list of dictionaries** — one dictionary per detected face.


In [ ]:
# Capture all the emotions detected in the image
captured_emotions = emo_detector.detect_emotions(input_image)

# Print the detected emotion information
print(captured_emotions)

**Example output:**
```python
[{'box': [223, 215, 437, 556],
  'emotions': {'angry': 0.0, 'disgust': 0.0, 'fear': 0.0, 'happy': 0.12,
               'sad': 0.0, 'surprise': 0.0, 'neutral': 0.88}}]
```
- `box` → `[x, y, width, height]` of the face bounding box.
- `emotions` → probability scores (0 to 1) for each emotion class. Here, **neutral (0.88)**
  is the dominant emotion.


## Step 5: Draw a Bounding Box Around the Face

We extract the `box` and `emotions` of the **first detected face** (`captured_emotions[0]`).

`cv2.rectangle()` draws a rectangle on the image:
- `(bounding_box[0], bounding_box[1])` → top-left corner (x, y)
- `(bounding_box[0] + bounding_box[2], bounding_box[1] + bounding_box[3])` → bottom-right
  corner (x + width, y + height)
- `(0, 255, 0)` → color in **BGR** = green
- `2` → line thickness in pixels


In [ ]:
# Make a bounding box around the face using cv2.rectangle to draw a green rectangle
bounding_box = captured_emotions[0]["box"]
emotions = captured_emotions[0]["emotions"]

cv2.rectangle(
    input_image,
    (bounding_box[0], bounding_box[1]),
    (bounding_box[0] + bounding_box[2], bounding_box[1] + bounding_box[3]),
    (0, 255, 0),
    2,
)

## Step 6: Find the Top Emotion and Label It

`emo_detector.top_emotion(image)` re-runs detection internally and directly returns:
- `emotion_name` → the emotion with the highest probability (e.g. `'neutral'`)
- `score` → its probability value (e.g. `0.88`)

> 💡 **Note:** The `if(score==np.max(score))` check is logically redundant here since
> `score` is already a single float value returned as the top score (not an array), so this
> condition is always `True`. It's harmless, but in a cleaner version you could just do
> `emotion = emotion_name` directly. (Good to mention if asked in an interview — shows you
> understand the code, not just run it.)

`cv2.putText()` then writes this emotion label below the bounding box:
- Position: just below the bottom of the box (`y + height + 30`)
- Font: `cv2.FONT_HERSHEY_SIMPLEX`, scale `1.5`
- Color: `(0, 256, 0)` → green (note: valid range is 0–255, 256 still works but 255 is
  technically correct)
- Thickness: `4`
- `cv2.LINE_AA` → anti-aliased (smoother) text edges


In [ ]:
import numpy as np

# Add Score to Bounding Box
# Store the top emotion in 'emotion_name' and its confidence in 'score'
emotion_name, score = emo_detector.top_emotion(input_image)

# Select the highest score emotion
if score == np.max(score):
    emotion = "{}".format(emotion_name)
    print("detected_emotion:", emotion_name)

# cv2.putText() draws a text string on the image
cv2.putText(
    input_image,
    emotion,
    (bounding_box[0], bounding_box[1] + bounding_box[3] + 30),
    cv2.FONT_HERSHEY_SIMPLEX,
    1.5,
    (0, 256, 0),
    4,
    cv2.LINE_AA,
)

## Step 7: Save the Annotated Image

`cv2.imwrite()` writes the modified image (with box + label) to disk as `facial_emotion.jpg`.


In [ ]:
# Save the result in a new image file
cv2.imwrite("facial_emotion.jpg", input_image)

## Step 8: Display the Result in the Notebook

We use `matplotlib.image` to read the saved image back (this time matplotlib reads it
correctly for display purposes) and `plt.imshow()` to show it inline.

`plt.axis('off')` hides the x/y pixel axes for a cleaner look.

> ⚠️ **Common gotcha:** OpenCV stores images in **BGR**, while matplotlib expects **RGB**.
> Since we *saved* the file with `cv2.imwrite` and *re-read* it with `mpimg.imread`, the
> colors will actually display correctly here. But if you ever skip the save/re-read step
> and try `plt.imshow(input_image)` directly on a cv2-loaded image, colors will look
> swapped (blue/red reversed) — you'd need
> `cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)` first. This is a classic interview question!


In [ ]:
import matplotlib.image as mpimg

# Read the saved image file using matplotlib's image module
result_image = mpimg.imread('facial_emotion.jpg')

# Create a new figure with size 10x10
plt.figure(figsize=(10, 10))

# Display the image
plt.imshow(result_image)
print("Emotion :", emotion_name)

# Remove the axis labels
plt.axis('off')
plt.show()

## Quick Interview Talking Points

| Question | Short Answer |
|---|---|
| What is FER? | A Python library for Facial Expression Recognition built on a pre-trained CNN model. |
| Why `mtcnn=True`? | Uses MTCNN instead of Haar Cascade for more robust/accurate face detection. |
| Why BGR vs RGB matters? | OpenCV reads/writes images in BGR; matplotlib/most other tools expect RGB. |
| What does `detect_emotions()` return? | A list of dicts, one per face, each with a `box` and an `emotions` probability dictionary. |
| What are the 7 emotion classes? | angry, disgust, fear, happy, sad, surprise, neutral. |
| What's `top_emotion()` for? | A convenience method that directly returns the single highest-confidence emotion + its score. |
| How would you handle multiple faces? | Loop over `captured_emotions` (a list) instead of indexing only `[0]`. |
| How would you improve this code? | Add error handling for "no face detected", loop for multiple faces, use RGB conversion directly instead of save/reload, and avoid the redundant `if score == np.max(score)` check. |
